In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

from latex_table import linear_regression

In [2]:
# Directories
PROJECT = "C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"
DATA = os.path.join(PROJECT, "Data")
FIGURES = os.path.join(PROJECT, "Figures")

In [3]:
# Load results from trading_daily notebook
results_file = os.path.join(DATA, 'trading_daily_results.pkl')
with open(results_file, 'rb') as f:
    results = pickle.load(f)

portfolios = results['portfolios']
portfolios_filtered = results['portfolios_filtered']
ff_factors = results['ff_factors']
prediction_columns = results['prediction_columns']
MODELS = results['MODELS']

print(f"Loaded results with {len(prediction_columns)} models")
print(f"Models: {list(MODELS.keys())}")

Loaded results with 2 models
Models: ['lr', 'lr_all']


# Time series tests (Fama-French regressions)

In [4]:
# Prepare the factor data
ff_factors_daily = ff_factors.copy() * 100  # Convert to percentage
ff_factors_daily['const'] = 1

# Use minimum 10 stocks portfolios as the main results
portfolios_to_test = portfolios_filtered[10]

# Calculate excess returns (long-short) for each portfolio and convert to percentage
portfolio_returns = {}
for pred_col in prediction_columns:
    if pred_col not in portfolios_to_test:
        continue
    portfolio = portfolios_to_test[pred_col]
    ls_ret = portfolio['long_ret'].fillna(0) - portfolio['short_ret'].fillna(0)
    portfolio_returns[pred_col] = ls_ret * 100  # Convert to percentage

# Merge all portfolio returns with factors
returns_df = pd.DataFrame(portfolio_returns)
returns_df.index.name = 'Date'

# Merge with factors
data = returns_df.merge(ff_factors_daily, left_index=True, right_index=True, how='inner')

print(f"Prepared regression data with {len(data)} observations")

Prepared regression data with 2768 observations


In [5]:
print("Running time series regressions...")
print("=" * 80)

# Dictionary to store all regression results
regression_results = {
    'CAPM': [],
    'FF3': [],
    'FF5': [],
    'FF6': []
}

# Run regressions for each prediction column
cols_to_test = [c for c in prediction_columns if c in data.columns]

for pred_col in cols_to_test:
    y = data[pred_col]
    
    # CAPM: alpha + beta * (Mkt-RF) + RF
    X_capm = data[['const', 'Mkt-RF']]
    model_capm = sm.OLS(y, X_capm).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['CAPM'].append(model_capm)
    
    # FF3: alpha + Mkt-RF + SMB + HML
    X_ff3 = data[['const', 'Mkt-RF', 'SMB', 'HML']]
    model_ff3 = sm.OLS(y, X_ff3).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['FF3'].append(model_ff3)
    
    # FF5: alpha + Mkt-RF + SMB + HML + RMW + CMA
    X_ff5 = data[['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
    model_ff5 = sm.OLS(y, X_ff5).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['FF5'].append(model_ff5)
    
    # FF6: FF5 + Mom
    X_ff6 = data[['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom']]
    model_ff6 = sm.OLS(y, X_ff6).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
    regression_results['FF6'].append(model_ff6)

print("Regressions complete!")
print(f"Total regressions run: {len(regression_results) * len(cols_to_test)}")

Running time series regressions...
Regressions complete!
Total regressions run: 8


In [6]:
# Create LaTeX tables for each factor model

# Define variable names for better display
var_names = {
    'const': 'Alpha',
    'Mkt-RF': 'Mkt-RF',
    'SMB': 'SMB',
    'HML': 'HML',
    'RMW': 'RMW',
    'CMA': 'CMA',
    'Mom': 'Mom'
}

# Column names from MODELS dict
cols_to_test = [c for c in prediction_columns if c in data.columns]
col_names_short = [next((m['name'] for m in MODELS.values() if m['col'] == c), c) for c in cols_to_test]

print("\n" + "=" * 80)
print("GENERATING LATEX TABLES (Returns already in percentage)")
print("=" * 80)

# Store tables for LaTeX export
latex_tables = {}

# CAPM Table
print("\n1. CAPM Model")
print("-" * 80)
tbl_capm = linear_regression(regression_results['CAPM'])
tbl_capm.set_var_list(['const', 'Mkt-RF'])
tbl_capm.rename_variables(var_names)
tbl_capm.rename_columns(col_names_short)
tbl_capm.aux_stat = 't'
tbl_capm.render(R2=True, obs=True)
latex_tables['CAPM'] = tbl_capm
print(tbl_capm.tbl.to_string())

# FF3 Table
print("\n2. Fama-French 3-Factor Model")
print("-" * 80)
tbl_ff3 = linear_regression(regression_results['FF3'])
tbl_ff3.set_var_list(['const', 'Mkt-RF', 'SMB', 'HML'])
tbl_ff3.rename_variables(var_names)
tbl_ff3.rename_columns(col_names_short)
tbl_ff3.aux_stat = 't'
tbl_ff3.render(R2=True, obs=True)
latex_tables['FF3'] = tbl_ff3
print(tbl_ff3.tbl.to_string())

# FF5 Table
print("\n3. Fama-French 5-Factor Model")
print("-" * 80)
tbl_ff5 = linear_regression(regression_results['FF5'])
tbl_ff5.set_var_list(['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA'])
tbl_ff5.rename_variables(var_names)
tbl_ff5.rename_columns(col_names_short)
tbl_ff5.aux_stat = 't'
tbl_ff5.render(R2=True, obs=True)
latex_tables['FF5'] = tbl_ff5
print(tbl_ff5.tbl.to_string())

# FF6 Table
print("\n4. Fama-French 6-Factor Model (FF5 + Momentum)")
print("-" * 80)
tbl_ff6 = linear_regression(regression_results['FF6'])
tbl_ff6.set_var_list(['const', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom'])
tbl_ff6.rename_variables(var_names)
tbl_ff6.rename_columns(col_names_short)
tbl_ff6.aux_stat = 't'
tbl_ff6.render(R2=True, obs=True)
latex_tables['FF6'] = tbl_ff6
print(tbl_ff6.tbl.to_string())


GENERATING LATEX TABLES (Returns already in percentage)

1. CAPM Model
--------------------------------------------------------------------------------
         Linear Regression Linear Regression (All Features)
Alpha        0.21THREESTAR                    0.25THREESTAR
                    (7.57)                          (11.62)
Mkt-RF      -1.19THREESTAR                            -0.03
                  (-23.65)                          (-1.30)
R2                    0.51                             0.00
N                    2,768                            2,768

2. Fama-French 3-Factor Model
--------------------------------------------------------------------------------
         Linear Regression Linear Regression (All Features)
Alpha        0.20THREESTAR                    0.25THREESTAR
                    (8.35)                          (11.57)
Mkt-RF      -1.06THREESTAR                            -0.02
                  (-29.33)                          (-0.81)
SMB         -1.